# ThreadCraft — Garment Classifier · Step 1: Data Cleaning

**Run on Kaggle with the CPU accelerator** — this is pure data wrangling and needs no GPU, and your GPU quota is better spent on `02_train.ipynb`. Use **Save & Run All (Commit)** so it runs headlessly.

**Source:** [`ashraq/fashion-product-images-small`](https://huggingface.co/datasets/ashraq/fashion-product-images-small) — a cleaned HF mirror of the Kaggle Fashion Product Images dataset.

Why this source rather than raw Kaggle:
- `load_dataset(...)` works with **no Kaggle dataset credentials** for the data itself
- Already 44,072 clean rows (the raw Kaggle `styles.csv` has malformed rows that crash `pd.read_csv`)
- Images are a native HF `Image` feature — no unzipping, no filename↔row matching
- MIT-licensed metadata, 60×80 px images — small enough to fine-tune in well under an hour on a T4

**Output:** a cleaned, stratified-split `DatasetDict` pushed to *your* HF account, which `02_train.ipynb` loads in one line.

## Why `articleType` restricted to Apparel

The raw dataset covers everything a fashion retailer sells — watches, handbags, deodorant, shoes. ThreadCraft is a **tailoring** platform, so the classifier's job is: *given a customer's uploaded reference photo, which garment is this?* — and the answer should map onto ThreadCraft's own cloth-type catalogue (T-shirt, Shirt, Dress, Trousers, Kurta, Saree Blouse, Salwar Kameez, Skirt).

So this notebook:
1. Filters to `masterCategory == 'Apparel'` — drops watches/bags/fragrance/footwear entirely
2. Uses **`articleType`** as the label (Tshirts, Shirts, Kurtas, Dresses, Trousers, Skirts, Sarees, …), which lines up with the platform's cloth types
3. **Drops** classes below a minimum count rather than lumping them into a meaningless `Other` bucket

Using the full 141-class `articleType` across all categories would mean a ~7,065:1 imbalance and a model that mostly predicts watches — good-looking accuracy, useless for this product. Set `RESTRICT_TO_APPAREL = False` if you want the everything-included variant as a comparison for your report.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
HF_USERNAME = "your-hf-username"  # <-- CHANGE THIS to your Hugging Face username

SOURCE_DATASET = "ashraq/fashion-product-images-small"
CLEANED_REPO_ID = f"{HF_USERNAME}/threadcraft-garments-cleaned"

RESTRICT_TO_APPAREL = True    # see the markdown above
TARGET_COLUMN = "articleType"
MIN_EXAMPLES_PER_CLASS = 100  # classes rarer than this are dropped entirely

VAL_FRACTION = 0.10
TEST_FRACTION = 0.10
RANDOM_SEED = 42
PUSH_TO_HUB = True

In [ ]:
!pip install -q -U datasets huggingface_hub scikit-learn

## Authenticate

Your HF token must be a **Kaggle Secret named `HF_TOKEN`** (Add-ons → Secrets), never pasted into a cell — Kaggle notebooks are frequently public, and a leaked write token gives anyone push access to your HF account. Setup steps: `docs/deployment/kaggle-huggingface-guide.md`.

In [ ]:
import os

from huggingface_hub import login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"Not on Kaggle or secret missing ({e}). Falling back to the HF_TOKEN env var.")
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
elif PUSH_TO_HUB:
    raise RuntimeError("No HF_TOKEN available but PUSH_TO_HUB is True. Add the Kaggle secret first.")

## 1. Load and inspect

In [ ]:
from datasets import load_dataset

raw = load_dataset(SOURCE_DATASET, split="train")
print(raw)
print()
print("Features:", {k: str(v) for k, v in raw.features.items()})

In [ ]:
import pandas as pd

# Metadata only — dropping the image column keeps EDA fast and memory-light.
# Decoding 44k PIL images just to count labels would be pointless.
meta = raw.remove_columns("image").to_pandas()
print(f"Total rows: {len(meta):,}")
meta.head()

In [ ]:
print("Nulls per column:")
print(meta.isnull().sum())
print()
print("masterCategory distribution:")
print(meta["masterCategory"].value_counts())

## 2. Filter and clean

1. Restrict to Apparel (if configured)
2. Drop rows with a null target
3. Drop exact duplicate metadata rows
4. Drop classes below `MIN_EXAMPLES_PER_CLASS` — a class with 3 examples can be neither learned nor stratified across three splits

In [ ]:
work = meta.copy()
start_rows = len(work)

if RESTRICT_TO_APPAREL:
    work = work[work["masterCategory"] == "Apparel"]
    print(f"Apparel filter:      {start_rows:,} -> {len(work):,} rows")

before = len(work)
work = work[work[TARGET_COLUMN].notnull()]
print(f"Null-target filter:  {before:,} -> {len(work):,} rows")

before = len(work)
work = work[~work.duplicated(keep="first")]
print(f"Duplicate filter:    {before:,} -> {len(work):,} rows")

print(f"\nDistinct {TARGET_COLUMN} values before rare-class pruning: {work[TARGET_COLUMN].nunique()}")

In [ ]:
counts = work[TARGET_COLUMN].value_counts()
print(f"Full {TARGET_COLUMN} distribution:")
print(counts.to_string())

keep_classes = counts[counts >= MIN_EXAMPLES_PER_CLASS].index.tolist()
dropped = counts[counts < MIN_EXAMPLES_PER_CLASS]

print(f"\nKeeping {len(keep_classes)} classes with >= {MIN_EXAMPLES_PER_CLASS} examples.")
print(f"Dropping {len(dropped)} rare classes ({dropped.sum():,} rows): {dropped.index.tolist()}")

In [ ]:
before = len(work)
work = work[work[TARGET_COLUMN].isin(keep_classes)]
print(f"Rare-class filter:   {before:,} -> {len(work):,} rows")

final_counts = work[TARGET_COLUMN].value_counts()
imbalance = final_counts.max() / final_counts.min()
print(f"\nFinal: {len(work):,} rows across {len(final_counts)} classes")
print(f"Largest class:  {final_counts.idxmax()} ({final_counts.max():,})")
print(f"Smallest class: {final_counts.idxmin()} ({final_counts.min():,})")
print(f"Imbalance ratio: {imbalance:.1f}:1  <- report macro-F1, not just accuracy")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, max(4, len(final_counts) * 0.25)))
final_counts.sort_values().plot(kind="barh", ax=ax, color="#8B6B4A")
ax.set_title(f"{TARGET_COLUMN} distribution after cleaning ({len(work):,} rows)")
ax.set_xlabel("examples")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=120)  # keep this figure for the dissertation
plt.show()

## 3. Stratified train / validation / test split

Stratified so every class keeps its proportion across all three splits — with this much imbalance a plain random split could leave a rare class entirely absent from validation.

In [ ]:
from sklearn.model_selection import train_test_split

labels = sorted(work[TARGET_COLUMN].unique())
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}

# Positional indices into the ORIGINAL `raw` dataset, so images stay aligned with labels.
original_positions = work.index.to_numpy()
strata = work[TARGET_COLUMN].to_numpy()

train_pos, temp_pos, _, temp_y = train_test_split(
    original_positions,
    strata,
    test_size=VAL_FRACTION + TEST_FRACTION,
    stratify=strata,
    random_state=RANDOM_SEED,
)
val_pos, test_pos, _, _ = train_test_split(
    temp_pos,
    temp_y,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),
    stratify=temp_y,
    random_state=RANDOM_SEED,
)

print(f"Train: {len(train_pos):,}   Val: {len(val_pos):,}   Test: {len(test_pos):,}")
assert len(set(train_pos) & set(val_pos)) == 0, "train/val overlap"
assert len(set(train_pos) & set(test_pos)) == 0, "train/test overlap"
assert len(set(val_pos) & set(test_pos)) == 0, "val/test overlap"
print("No overlap between splits.")

In [ ]:
from datasets import ClassLabel, DatasetDict

class_label = ClassLabel(names=labels)
label_lookup = dict(zip(meta.index, meta[TARGET_COLUMN]))


def build_split(positions):
    subset = raw.select(positions.tolist())
    label_ids = [label2id[label_lookup[p]] for p in positions]
    subset = subset.add_column("label", label_ids).cast_column("label", class_label)
    # Keep only what training and verification need — smaller repo, faster download.
    keep = {"image", "label", "articleType", "subCategory", "masterCategory", "baseColour", "gender"}
    drop = [c for c in subset.column_names if c not in keep]
    return subset.remove_columns(drop)


dataset_dict = DatasetDict(
    {
        "train": build_split(train_pos),
        "validation": build_split(val_pos),
        "test": build_split(test_pos),
    }
)
dataset_dict

## 4. Sanity check — do the images actually match their labels?

Always eyeball this. A silent off-by-one in the index alignment above would produce a dataset that trains happily to a meaningless result, and the loss curve alone will not tell you.

In [ ]:
sample = dataset_dict["train"].shuffle(seed=RANDOM_SEED).select(range(10))
fig, axes = plt.subplots(2, 5, figsize=(14, 7))
for ax, row in zip(axes.flatten(), sample):
    ax.imshow(row["image"])
    ax.set_title(f"{id2label[row['label']]}\n({row['articleType']})", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig("sample_images.png", dpi=120)
plt.show()

# The mapped label must equal the original articleType in every row.
mismatches = [r for r in sample if id2label[r["label"]] != r["articleType"]]
print(f"Label/articleType mismatches in sample: {len(mismatches)} (must be 0)")
assert not mismatches, "Label alignment is broken — do not train on this."

## 5. Push the cleaned dataset to the Hub

In [ ]:
import json

with open("label2id.json", "w") as f:
    json.dump(label2id, f, indent=2)
print(json.dumps(label2id, indent=2))

In [ ]:
if PUSH_TO_HUB:
    # Wrapped so a network blip on the push doesn't discard the whole run.
    # /kaggle/working persists across a commit, so you can re-push from there
    # without re-running the cleaning.
    try:
        dataset_dict.push_to_hub(CLEANED_REPO_ID, private=False)
        from huggingface_hub import HfApi

        HfApi(token=HF_TOKEN).upload_file(
            path_or_fileobj="label2id.json",
            path_in_repo="label2id.json",
            repo_id=CLEANED_REPO_ID,
            repo_type="dataset",
        )
        print(f"\nPushed: https://huggingface.co/datasets/{CLEANED_REPO_ID}")
    except Exception as e:
        print(f"PUSH FAILED: {e}")
        dataset_dict.save_to_disk("/kaggle/working/threadcraft-garments-cleaned")
        print("Saved to /kaggle/working/threadcraft-garments-cleaned instead — re-push from there.")
else:
    dataset_dict.save_to_disk("/kaggle/working/threadcraft-garments-cleaned")
    print("PUSH_TO_HUB is False — saved locally only.")

## Record these numbers for your dissertation

In [ ]:
print("=" * 62)
print("DATA PREPARATION SUMMARY — garment classifier")
print("=" * 62)
print(f"Source dataset        : {SOURCE_DATASET}")
print(f"Raw rows              : {len(meta):,}")
print(f"Restricted to Apparel : {RESTRICT_TO_APPAREL}")
print(f"Rows after cleaning   : {len(work):,}")
print(f"Target column         : {TARGET_COLUMN}")
print(f"Classes kept          : {len(labels)} (>= {MIN_EXAMPLES_PER_CLASS} examples each)")
print(f"Classes dropped       : {len(dropped)}")
print(f"Imbalance ratio       : {imbalance:.1f}:1")
print(f"Train / Val / Test    : {len(train_pos):,} / {len(val_pos):,} / {len(test_pos):,}")
print(f"Split strategy        : stratified, seed={RANDOM_SEED}")
print("=" * 62)
print("\nNext: run 02_train.ipynb with the GPU T4 x2 accelerator.")